# Building a Multi-Agent Research Pipeline with LangGraph

This notebook is one continuous lesson built around a single script: an **orchestrator–worker research pipeline** that uses LangGraph to plan a research topic, fan it out to several parallel "researcher" agents, and stitch their findings back together.

We won't walk through the script top to bottom. Instead we'll build the same system from scratch, introducing each LangGraph/LangChain concept exactly when the growing pipeline needs it — the same order you'd actually *design* this system in, not the order the file happens to declare things in.

**Where we're headed**, in one sentence: a single LLM call decides *what to research*, LangGraph's `Send` API dynamically spins up one worker per research angle, each worker calls the LLM independently, and a final node merges everything into a report.

Along the way we'll hit: `TypedDict` state schemas, reducers (`Annotated[..., add]`), rate limiting, retry logic with `tenacity`, structured output via Pydantic, the `Send` primitive for map-reduce style fan-out, and conditional edges.

> 💡 Throughout, look for **⚠️ Bug watch** callouts — these flag real issues in the original script. Try to spot them yourself before reading the explanation.

## 1. Why "orchestrator–worker" instead of one big prompt?

Imagine you ask an LLM: *"Research the impact of AI on healthcare and write me a report."* You could just send that as one giant prompt. But a single LLM call has to simultaneously decide **what** to look into and **write about all of it at once** — it tends to be shallow, because breadth and depth compete for the same context window and the same "attention."

The **orchestrator–worker** pattern (sometimes called *plan-and-execute* or *fan-out/fan-in*) splits this into three roles:

1. **Orchestrator** — a "planner" LLM call that decides *which distinct sub-questions* are worth investigating.
2. **Workers** — one LLM call *per sub-question*, run independently (and in this script, this happens in parallel), each free to go deep on just its slice.
3. **Synthesizer** — a final step that merges all the worker outputs into one coherent deliverable.

This is a genuinely different shape of computation from a normal chain (`prompt → LLM → output`): the *number* of worker steps isn't fixed in advance — it depends on what the orchestrator decides. That dynamic branching is exactly what LangGraph's `Send` API is built for, and it's the centerpiece of this lesson.

We'll build the state model first, since in LangGraph *everything* — nodes, edges, parallelism — is defined in terms of how it reads and writes a shared state object.

In [ ]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langgraph.types import Send
from typing import TypedDict, List, Literal, Annotated
from operator import add
from pydantic import BaseModel, Field
from langchain_core.rate_limiters import InMemoryRateLimiter

load_dotenv()

A few notes on these imports before moving on:

- `ChatGroq` — LangChain's chat-model wrapper around Groq's API (Groq hosts open-weight models like `openai/gpt-oss-20b` and serves them extremely fast thanks to their custom LPU hardware).
- `StateGraph`, `START`, `END` — the core building blocks of a LangGraph graph. `StateGraph` is the graph itself; `START`/`END` are sentinel nodes marking entry and exit.
- `Send` — the primitive that lets a single node dynamically dispatch *N* parallel invocations of another node. This is the key to fan-out.
- `TypedDict`, `Annotated` — used to define the *shape* of the graph's shared state, and to attach **reducers** to individual fields (more in a moment).
- `operator.add` — used as a reducer function; when workers write to the same state key concurrently, `add` tells LangGraph *how* to combine their contributions (list concatenation, here) instead of one overwriting another.
- `InMemoryRateLimiter` — throttles how often LangChain will actually call the underlying LLM, independent of any retry logic.
- `load_dotenv()` — loads your `GROQ_API_KEY` (and friends) from a `.env` file into the environment, so `ChatGroq` can pick it up automatically.

⚠️ **Bug watch:** `Literal` is imported but never used anywhere in the script. Harmless, but it's dead weight — a linter would flag it as an unused import.

## 2. Designing the shared state

Every LangGraph node is a plain function: `(state) -> partial_state_update`. LangGraph merges that partial update back into the graph's overall state after each node runs. So the *state schema* is the contract every node has to honor — it's worth designing carefully before writing any node logic.

This pipeline actually needs **two** state schemas, because the orchestrator/synthesizer operate on the *whole* research project, while each worker only needs to know about *its own* slice of it.

In [ ]:
class OverallState(TypedDict):
    research_topic : str
    sources        : List[str]
    worker_findings: Annotated[List[dict], add]
    final_report   : str

class WorkerState(TypedDict):
    source   : str
    worker_id: int
    research_topic: str

`OverallState` is the state that flows through the graph's main spine (orchestrator → synthesizer). `WorkerState` is a *narrower* view that each individual worker instance receives — it doesn't need `worker_findings` or `final_report` at all, just enough to do its one job.

The genuinely important line here is:

```python
worker_findings: Annotated[List[dict], add]
```

**Why does this need a reducer at all?** Because multiple workers are going to run *concurrently* and each is going to try to write to `worker_findings`. Without a reducer, LangGraph's default behavior for a repeated key is "last write wins" — worker 3's finding would silently overwrite worker 1's and worker 2's. That would silently destroy data with no error raised.

`Annotated[List[dict], add]` tells LangGraph: *"when more than one update touches this key in the same step, combine them using Python's `add` operator"* — and since these are lists, `add` means concatenation (`[a] + [b] = [a, b]`). So instead of overwriting, every worker's single-item list gets appended to a running list. This is the same "reducer" concept you'll see in `functools.reduce` or in Redux-style state management — a function that folds multiple updates into one.

Contrast this with `sources` and `final_report`, which have *no* `Annotated` wrapper — they use the implicit default reducer (overwrite). That's fine because only one node (the orchestrator, and the synthesizer respectively) ever writes to those keys.

## 3. Talking to the LLM reliably: rate limits and retries

Before writing any node, the script sets up two pieces of defensive infrastructure around LLM calls. It's worth understanding *why both exist*, because they solve two different problems that look similar on the surface.

In [ ]:
from tenacity import retry, retry_if_exception_type, wait_exponential, stop_after_attempt
from groq import RateLimitError

@retry(
    retry=retry_if_exception_type(RateLimitError),
    wait=wait_exponential(multiplier=1, min=2, max=30),
    stop=stop_after_attempt(5),
)
def safe_invoke(llm, prompt):
    return llm.invoke(prompt)

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.3,   # ~1 request every ~3s, tune to your TPM
    check_every_n_seconds=0.1,
    max_bucket_size=1,
)

**`InMemoryRateLimiter` is proactive** — it's attached to the `ChatGroq` model itself (we'll see that next), and it *paces* outgoing requests *before* they're sent, so you're less likely to hit a rate limit in the first place. `requests_per_second=0.3` caps you at roughly one call every 3 seconds; `max_bucket_size=1` means you can't "save up" bursts of allowance.

**`tenacity`'s `@retry` is reactive** — it wraps a function call and, *if* it still fails with a `RateLimitError` (e.g. because the rate limiter's pacing wasn't conservative enough, or a burst of concurrent workers slipped through), it retries with **exponential backoff**: wait ~2s, then ~4s, ~8s, ~16s, up to a 30s ceiling, giving up after 5 attempts total.

Why use both instead of just one? The rate limiter reduces *how often* you hit the problem; the retry logic handles it gracefully *when* it still happens anyway (which is likely here, since several workers will be calling the same rate-limited model concurrently via `Send`). This is a decorator pattern — `@retry(...)` wraps `safe_invoke` so that every call site (`safe_invoke(llm, prompt)`) automatically gets this resilience without repeating try/except boilerplate.

⚠️ **Bug watch:** `retry_if_exception_type(RateLimitError)` means *only* `RateLimitError` triggers a retry. Any other transient failure — a network timeout, a Groq 5xx error, a JSON-parsing failure inside structured output — will propagate immediately and crash the whole graph run, with no retry at all. That's a deliberate narrowing, but it's worth being aware of: in a longer-running pipeline you might want a broader set of retryable exceptions (or at least log-and-continue behavior for a single failed worker instead of killing the entire batch).

## 4. Getting the LLM to return *structured* data

The orchestrator's whole job is to turn a fuzzy topic string into a clean list of research sources. If we just asked the LLM to "list some sources" in free text, we'd then have to parse its prose back into a Python list — fragile and error-prone. Instead, we define the *exact shape* we want as a Pydantic model, and ask LangChain to force the model's output to match it.

In [ ]:
class ResearchPlan(BaseModel):
    sources   : List[str] = Field(description= "List of specific research sources/aspects to investigate.", max_length=5)
    reasoning : str = Field(description="Brief explanation of why these sources where chosen")

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0, rate_limiter=rate_limiter, reasoning_effort="low")

`ResearchPlan` is a normal Pydantic `BaseModel` — `sources` must be a list of strings (with `max_length=5`, i.e. at most 5 items), and `reasoning` must be a string. Each `Field(description=...)` isn't just documentation for humans — that description text is actually **injected into the schema sent to the LLM**, so the model sees *why* each field exists and what's expected of it.

`llm` here is the base `ChatGroq` model, configured once:
- `model="openai/gpt-oss-20b"` — an open-weight OpenAI model served via Groq.
- `temperature=0` — deterministic-as-possible output, appropriate for planning/extraction tasks where you don't want creative variance.
- `rate_limiter=rate_limiter` — every call through this model instance is paced by the limiter we just built.
- `reasoning_effort="low"` — a Groq-specific parameter that tunes how much internal "thinking" the model does before answering (trades latency/cost for quality).

Later, `plan_research` will call `llm.with_structured_output(ResearchPlan, method="json_schema")` to get a model that returns a *validated* `ResearchPlan` instance instead of raw text — LangChain handles asking the underlying model to emit JSON matching the schema, then parses and validates it against the Pydantic model for you.

⚠️ **Bug watch (comment/code mismatch):** the code that builds the structured-output model is preceded by the comment `# Option B: keep gpt-oss-120b but use json_mode instead of tool-calling for structured output`. But the actual model configured above is `openai/gpt-oss-20b` (not `120b`), and the method actually used is `"json_schema"` (not `"json_mode"`). This looks like a leftover comment from an earlier iteration of the script that wasn't updated when the code changed — a small thing, but the kind of stale comment that actively misleads whoever reads the code next (including future-you).

## 5. The orchestrator node

Now we can write the first actual graph node: a function that takes `OverallState` in, calls the structured-output model with a planning prompt, and returns a *partial* state update (just the `sources` key — LangGraph merges it into the full state for us).

In [ ]:
def plan_research(state: OverallState) -> OverallState:

    print("\n" + "=" * 70)
    print("Orchestrator: Planning research strategy")
    print("=" * 70)
    print(f"Topic: {state['research_topic']}\n")

    planner_llm = llm.with_structured_output(ResearchPlan, method="json_schema")
    prompt = f"""You are a research strategist planning a comprehensive investigation:

    Research Topic: {state['research_topic']}

    CRITICAL INSTRUCTION: Generate between 3-5 specific research sources or aspects to investigate.
    DO NOT generate more than 5 sources.

    Each source should be:
    - Specific and focused on a distinct aspect
    - Relevant to the overall topic.
    - Complementary to other sources (minimum overlap)
    - Concrete enough to guide targeted research

    Generate sources that will provide comprehensive coverage of the topic.
    """

    research_plan = safe_invoke(planner_llm, prompt)

    print(f"Orchestrator generated {len(research_plan.sources)} research sources")
    for i, source in enumerate(research_plan.sources, 1):
        print(f"{i}, {source}")
    print(f"\nReasoning: {research_plan.reasoning}")
    print("Preparing to dispatch to nodes ...")

    return {"sources": research_plan.sources}

A couple of things worth noticing here, beyond the mechanics:

- The function returns `{"sources": research_plan.sources}` — **not** the full state. This is the normal LangGraph convention: a node returns only the *keys it changed*, and LangGraph merges that into the running state using each key's reducer (plain overwrite for `sources`, since it has none).
- `safe_invoke(planner_llm, prompt)` routes the call through our retry wrapper, so a transient `RateLimitError` here gets retried automatically.
- The prompt itself is a plain multi-line f-string — nothing fancy, just clear natural-language instructions plus a few-shot-style list of "good" example sources (omitted above for brevity, present in the original) to anchor the model's sense of granularity.

⚠️ **Bug watch (typos, cosmetic):** in the original script this prompt contains a stray slash — `"...investigate/"` where a period was clearly intended — and two spelling typos, `"mroe"` (more) and, in the worker prompt later, `"compliacne"` (compliance) and `"thorugh"` (thorough). None of these will break the code, since they're just inside a string the LLM reads, but they're worth catching: typos in prompts are invisible to Python's syntax checker (the string is still valid) but can subtly confuse the model or just look unprofessional if these prompts are ever logged or reviewed.

⚠️ **Bug watch (also cosmetic):** one of the `print` calls in the original uses a capital-F f-string — `print(F"Orchestrator generated...")` — while every other f-string in the file uses lowercase `f`. Python accepts both, so this isn't a bug in the "will crash" sense, but inconsistent casing like this is usually a sign of copy-pasting from different sources, and it's the kind of thing a linter/formatter would flag as a style inconsistency.

## 6. Fanning out: the `Send` API

This is the heart of the pattern. The orchestrator produced a *variable-length* list of sources (anywhere from 3 to 5). We now need to run the worker logic **once per source, in parallel**, and LangGraph needs to know how to do that *without* us hard-coding "3 workers" or "5 workers" into the graph structure ahead of time.

This is exactly what `Send` is for. Instead of a node returning a state update, a special **conditional edge function** can return a *list of `Send` objects* — each one saying "dispatch to node X, with this specific state.

In [ ]:
def create_research_workers(state: OverallState) -> OverallState:
    print("DISPATCHER: Creating research workers dynamically...")

    return [
        Send("research_worker", {
            "source": source,
            "worker_id": i + 1,
            "research_topic": state["research_topic"]
        })
        for i, source in enumerate(state["sources"])
    ]

Each `Send("research_worker", {...})` says: *"invoke the node named `research_worker`, but instead of giving it the full graph state, give it exactly this dict"* — which matches the shape of `WorkerState`. Because this returns a **list**, LangGraph will schedule one invocation of `research_worker` per list item, and — crucially — run them **concurrently**. This is a map-reduce pattern: `create_research_workers` is the "map" step (turn 1 input into N tasks), and the `worker_findings` reducer we defined earlier is the "reduce" step (fold N outputs back into one list).

⚠️ **Bug watch (misleading type hint):** the function is annotated `-> OverallState`, but it doesn't actually return an `OverallState` (or a partial one) at all — it returns `List[Send]`. This works at runtime because Python type hints aren't enforced, and because LangGraph specifically recognizes this "conditional edge that returns `Send` objects" pattern. But if you were reading this signature cold, `-> OverallState` would actively mislead you about what the function does. A more honest annotation would be something like `-> list[Send]`.

Notice also that `create_research_workers` is **not registered as a graph node** — it's used as the *routing function* for a conditional edge (we'll see that in the graph-building section). That distinction matters: nodes update state, routing functions decide *where execution goes next* (and, in this special case, *how many parallel branches to create*).

## 7. The worker node

Each worker is a completely independent function invocation — it doesn't know how many siblings it has, and it can't see their output. It just receives its own `WorkerState` (one `source`, its `worker_id`, and the shared `research_topic`), does its research, and returns a single finding.

In [ ]:
def research_worker(state: WorkerState) -> OverallState:

    worker_id = state["worker_id"]
    source    = state["source"]

    print(f" WORKER {worker_id}: Researching '{source}'...")

    prompt = f"""You are a specialized researcher investigating:
    {state['research_topic']}

    Your specific focus area: {source}

    Conduct thorough research on this aspect and provide:

    1. KEY FINDINGS (3-5 specific points)
    2. DATA and STATISTICS
    3. INSIGHTS and ANALYSIS
    4. SOURCES and CREDIBILITY
    5. IMPLICATIONS

    Be specific, factual, and provide depth on this particular aspect.
    """

    response = safe_invoke(llm, prompt).content

    findings = {
        "worker_id": worker_id,
        "source": source,
        "content": response
    }

    print(f"WORKER {worker_id}: Research Complete\n")

    return {"worker_findings": [findings]}

Two design choices worth calling out:

1. **`response = safe_invoke(llm, prompt).content`** — note this uses the *plain* `llm`, not `planner_llm` with structured output. That's intentional here: the worker's job is to produce free-flowing research prose, not a rigid schema, so plain text generation is the right tool. Using structured output for *everything* just because you used it once elsewhere would be over-engineering.
2. **`return {"worker_findings": [findings]}`** — this returns a *single-item list* containing this worker's one finding. This is exactly why we needed the `add` reducer on `worker_findings`: when 3–5 workers each return `[findings]` concurrently, LangGraph concatenates them all into one combined list rather than the last worker's single item overwriting everyone else's.

This is also a nice moment to connect back to §2: `WorkerState` deliberately does *not* include `worker_findings` or `final_report`, because a worker has no business reading or writing the *aggregate* state — it only needs its own slice in, and produces one small piece out. Keeping the input state narrow is good practice: it makes each worker's contract easy to reason about in isolation.

## 8. The synthesizer node — and a subtle but important bug

The final node is supposed to take all the workers' findings and ask the LLM to weave them into one coherent report.

In [ ]:
def synthesize_report(state: OverallState) -> OverallState:
    print("=" * 70)
    print("SYNTHESIZER: Combining insights from all workers")
    print("=" * 70)

    print(f"Processing findings from {len(state['worker_findings'])} research sources")

    all_findings = "\n\n" + "=" * 70 + "\n\n"
    all_findings += "\n\n".join([
        f"RESEARCH AREA {f['worker_id']}: {f['source']}\n{'-'*70}\n{f['content']}"
        for f in state['worker_findings']
    ])

    prompt = f"""You are synthesizing a comprehensive research report on: {state['research_topic']}

    You have received detailed findings from {len(state['worker_findings'])} specialized researchers.

    RESEARCH FINDINGS:
    {all_findings}

    Create a cohesive, well structured research report (500-700 words).
    Write this as a unified, flowing report, NOT as separate sections from each researcher.
    """

    # final_report = safe_invoke(llm, prompt).content
    final_report = all_findings

    print("SYNTHESIZER: Final report complete\n")
    return {"final_report": final_report}

⚠️ **Bug watch — this is the biggest one in the script.** Look closely at the last few lines:

```python
# final_report = safe_invoke(llm, prompt).content
final_report = all_findings
```

The synthesis LLM call is **commented out**. A carefully constructed synthesis `prompt` is built — including all the worker findings and detailed instructions for how to merge them into a 500–700 word unified report — and then it's **never sent to the model**. Instead, `final_report` is just set directly to `all_findings`, the raw concatenation of each worker's unedited output with `RESEARCH AREA N:` headers between them.

The practical effect: the graph's advertised "synthesizer" step doesn't actually synthesize anything. What you get back as `result['final_report']` is essentially the same as if you'd skipped the synthesizer node entirely and just joined the worker outputs together. This is almost certainly leftover from debugging (a very common pattern: comment out the expensive/slow LLM call while iterating on everything *around* it, so you're not burning API calls or waiting on rate limits every run) — but if this script were shipped as-is, whoever calls it would silently get an unsynthesized document while believing they got a polished report.

⚠️ **Bug watch (smaller, stylistic):** the `prompt` string starts with an f-string opener followed immediately by a fourth quote character and then the word "You" — four quote characters in a row before the text begins. Since the triple-quote opens the string, that fourth quote becomes a literal character at the very start of the string content itself. It doesn't break anything (the string is still syntactically valid), but the resulting prompt text literally begins with a stray quotation mark, which is very likely an accidental extra keystroke rather than intentional formatting.

## 9. Assembling the graph

We now have three functions (`plan_research`, `research_worker`, `synthesize_report`) and one routing function (`create_research_workers`). Wiring them into a `StateGraph` is mostly bookkeeping — but the *order* in which you call `add_node` / `add_edge` mirrors the actual flow of execution, so it's worth reading top to bottom as a story.

In [ ]:
builder = StateGraph(OverallState)

builder.add_node("orchestrator", plan_research)
builder.add_node("research_worker", research_worker)
builder.add_node("synthesizer", synthesize_report)

builder.add_edge(START, "orchestrator")
builder.add_conditional_edges(
    "orchestrator",
    create_research_workers,
    ["research_worker"]
)
builder.add_edge("research_worker", "synthesizer")
builder.add_edge("synthesizer", END)

graph = builder.compile()

Reading this as a story:

- `StateGraph(OverallState)` — the graph is built around the *overall* schema; individual `Send` calls temporarily swap in `WorkerState` for a single node invocation.
- `add_node(name, fn)` — registers each function under a string name; that name is what edges and `Send` calls refer to. Note that `create_research_workers` is **not** added as a node — it's a routing function only, passed directly to `add_conditional_edges`.
- `add_edge(START, "orchestrator")` — execution always begins at the orchestrator.
- `add_conditional_edges("orchestrator", create_research_workers, ["research_worker"])` — after the orchestrator finishes, instead of going to one fixed next node, LangGraph calls `create_research_workers(state)`. Since that returns a list of `Send` objects, LangGraph dynamically creates one `"research_worker"` invocation per item. The third argument, `["research_worker"]`, tells LangGraph the set of possible destination nodes this edge could route to (useful for graph visualization/validation).
- `add_edge("research_worker", "synthesizer")` — **every** worker instance, once finished, has an edge into `synthesizer`. LangGraph automatically waits for *all* concurrently-dispatched workers to complete before running `synthesizer` — this is the "reduce" half of map-reduce; it's an implicit synchronization barrier, not something we had to code by hand.
- `add_edge("synthesizer", END)` — and we're done.
- `builder.compile()` — turns the declarative graph description into an executable `Runnable`.

This is a good moment to connect back to §6: the reason `add_conditional_edges` (rather than a plain `add_edge`) is required here is *specifically* because the number of downstream branches isn't known until runtime — it depends on how many `sources` the orchestrator produced. A plain `add_edge("orchestrator", "research_worker")` would only ever create a single worker invocation; `Send` is what turns one edge into N parallel edges.

## 10. Running it

Finally, the script picks a topic and invokes the compiled graph. `graph.invoke` takes an initial state dict matching (a subset of) `OverallState`, runs the whole graph to completion, and returns the final state.

In [ ]:
healthcare_topic = "The impact of artificial intelligence on healthcare delivery and patient outcomes"
environmental_topic = "Renewable energy adoption barriers in developing countries."

topic = healthcare_topic

print("=" * 70)
print(f"Topic: {topic}")
print("=" * 70)

result = graph.invoke({
    "research_topic": topic,
    "sources": [],
    "worker_findings": []
})

print("=" * 70)
print("FINAL SYNTHESIZED RESULT")
print("=" * 70)
print(result['final_report'])

A quick note on the initial state dict: `sources` and `worker_findings` are seeded as empty — `sources` will be overwritten wholesale by the orchestrator (no reducer, so plain replacement), while `worker_findings` starts as `[]` specifically so that the `add` reducer has something valid to concatenate onto as each worker's `[findings]` list comes in. `final_report` isn't seeded at all, which is fine since it's only ever written once, by the synthesizer, at the very end.

⚠️ **Bug watch (unused variable):** `environmental_topic` is defined but `topic` is hard-coded to `healthcare_topic` — the second topic is dead code as written. Not a bug exactly (maybe it's there for quick manual swapping during testing), but worth flagging since an unused, fully-formed variable sitting right next to the one that *is* used is an easy thing to trip over later (e.g. editing the wrong one and wondering why nothing changed).

**To actually run this yourself**, you'd need a `.env` file with a valid `GROQ_API_KEY`, and the `langchain-groq`, `langgraph`, `tenacity`, and `groq` packages installed. Running it live would consume Groq API quota, so we've kept this notebook as an explanation-and-reference rather than executing the LLM calls directly — but every code cell above is copy-paste runnable in sequence, in your own environment, once your API key is set.

## 11. Synthesis: how this connects to the bigger picture

If you've worked with plain LangChain chains before (`prompt | llm | parser`), the jump here is from a **linear pipeline** to a **graph with dynamic branching and an implicit join**. A few ideas from this notebook generalize well beyond this specific script:

- **State + reducers** are how LangGraph handles concurrency safely. Any time multiple branches might write to the same key, ask: "overwrite, or combine?" — and reach for `Annotated[T, reducer_fn]` when the answer is "combine."
- **`Send` is the tool for "I don't know how many parallel tasks I need until runtime."** Whenever a variable-length list needs to become N independent LLM calls (or tool calls, or sub-agents), this is the pattern.
- **Rate limiting and retries are complementary, not redundant** — one is preventative pacing, the other is reactive recovery. Production LLM pipelines generally want both.
- **Structured output is for extraction/planning steps where downstream code needs to consume the result programmatically**; free-text generation is still the right tool for open-ended writing tasks (as the worker node shows). Don't force everything into a schema.
- **A conditional-edge routing function and a graph node are different things**, even though both are plain Python functions taking `state` — nodes update state, routing functions decide (or, with `Send`, *multiply*) what happens next.

This maps directly onto the broader "orchestrator/planner + parallel sub-agents + aggregator" shape you'll see across agent frameworks generally (not just LangGraph) — it's one of the most common patterns for turning a single vague request into a structured, parallelized, then re-combined body of work.

## 12. Exercises

Try these directly against the code above (mentally, or in your own environment with an API key):

1. **Fix the synthesizer bug.** Uncomment the real LLM call in `synthesize_report` and remove the `final_report = all_findings` fallback. What do you expect to change about the final output's length and tone compared to the raw concatenation?
2. **Broaden the retry policy.** Modify `safe_invoke`'s `@retry` decorator so it also retries on a generic `Exception` (or a more targeted set like connection errors), not just `RateLimitError`. What's the risk of retrying on *too broad* a set of exceptions?
3. **Add a `max_workers` guard.** Right now, `create_research_workers` dispatches one `Send` per source unconditionally (up to 5, per the Pydantic `max_length`). Add a runtime check that logs a warning (or truncates) if somehow more than 5 sources come back from the planner, so a schema-validation slip doesn't silently blow past your rate limiter.
4. **Give each worker its own `WorkerState` field for errors.** Right now if `safe_invoke` inside `research_worker` exhausts its retries, the exception propagates and kills the *entire* graph run — including the work of every other worker that succeeded. Sketch (in comments, or actual code) how you'd catch that exception inside `research_worker` and return a `findings` dict with an `"error"` field instead, so the synthesizer can gracefully skip or flag failed workers rather than the whole run crashing.
5. **Swap the topic.** Change `topic = healthcare_topic` to use `environmental_topic` instead (this finally makes that variable not dead code!) and predict how the *shape* of the orchestrator's generated `sources` list might differ for a barriers-to-adoption topic versus a delivery-and-outcomes topic.

## 13. Summary of concepts covered

- **Orchestrator–worker (plan-and-execute) pattern** — splitting a broad task into a planning step, N parallel execution steps, and a merge step.
- **`TypedDict` state schemas** (`OverallState`, `WorkerState`) — defining the shape of data flowing through a LangGraph graph, including narrower per-node state views.
- **Reducers via `Annotated[T, fn]`** — controlling how concurrent writes to the same state key are combined (`operator.add` for list concatenation), versus the default overwrite behavior.
- **`InMemoryRateLimiter`** — proactive request pacing attached directly to a chat model.
- **`tenacity` retry decorators** (`retry_if_exception_type`, `wait_exponential`, `stop_after_attempt`) — reactive, exponential-backoff recovery from specific exception types.
- **Structured output** via Pydantic `BaseModel` + `with_structured_output(schema, method=...)` — forcing LLM output into a validated schema, useful for planning/extraction steps that feed downstream code.
- **`Send` for dynamic fan-out** — turning a runtime-determined list into N parallel node invocations, the core mechanism behind map-reduce-style graphs in LangGraph.
- **Conditional edges vs. graph nodes** — routing functions (used with `add_conditional_edges`) decide *where* execution goes (and, with `Send`, *how many* branches to create); nodes update state.
- **Implicit join/synchronization** — when multiple `Send`-dispatched branches all point to the same downstream node (`synthesizer`), LangGraph waits for all of them before proceeding — no manual barrier code required.
- **Bugs/smells spotted along the way:** an unused `Literal` import, a stale comment describing an older model/method than what's actually configured, several prompt typos, a capitalization inconsistency in an f-string, a misleading `-> OverallState` type hint on a function that returns `list[Send]`, a commented-out LLM call that silently downgrades the "synthesizer" to a plain concatenation, an accidental quadruple-quote at the start of a prompt string, and an unused `environmental_topic` variable.